# FRED Data Preprocessing
Cleaning the raw risk-free rate and CPI series: filling gaps, converting units, and aligning to trading dates.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
RAW_DATA_PATH = Path('../../data/raw/fred')
PROCESSED_DATA_PATH = Path('../../data/processed/fred')

## Load raw data

In [3]:
rf = pd.read_csv(RAW_DATA_PATH / 'fred_risk_free_rate_raw.csv', index_col='date', parse_dates=True)
cpi = pd.read_csv(RAW_DATA_PATH / 'fred_cpi_raw.csv', index_col='date', parse_dates=True)

print("Risk-free rate shape:", rf.shape)
print("CPI shape:", cpi.shape)

Risk-free rate shape: (2088, 1)
CPI shape: (96, 1)


## Clean the risk-free rate series
FRED's Treasury series has gaps on weekends/holidays. We reindex to every calendar day and forward-fill,
since a rate stays constant until the next published update.

In [4]:
rf_filled = rf.asfreq('D').ffill()

print("Missing values after ffill:", rf_filled['risk_free_rate_pct'].isna().sum())
rf_filled.head(10)

Missing values after ffill: 1


,risk_free_rate_pct
date,
2018-01-01,NaN
2018-01-02,1.44
2018-01-03,1.41
2018-01-04,1.41
2018-01-05,1.39
2018-01-06,1.39
2018-01-07,1.39
2018-01-08,1.45
2018-01-09,1.44


## Convert annualized percentage to decimal
FRED reports this as an annualized percent (e.g. 5.25 = 5.25%). Convert to decimal for use in Sharpe ratio math.

In [5]:
rf_filled['risk_free_rate_decimal'] = rf_filled['risk_free_rate_pct'] / 100
rf_filled.head()

,risk_free_rate_pct,risk_free_rate_decimal
date,,
2018-01-01,NaN,NaN
2018-01-02,1.44,0.0144
2018-01-03,1.41,0.0141
2018-01-04,1.41,0.0141
2018-01-05,1.39,0.0139


## Align to trading dates
Replace this with the actual trading date index from the price data notebook once available.
For now this uses US business days as a placeholder.

In [6]:
# Align FRED data to the same trading dates used by the yfinance price dataset.
YFINANCE_PROCESSED_DATA_PATH = Path('../../data/processed/yfinance')

trading_dates = pd.read_csv(
    YFINANCE_PROCESSED_DATA_PATH / 'yfinance_adjusted_close_clean.csv',
    index_col=0,
    parse_dates=True
).index

rf_aligned = rf_filled.reindex(trading_dates).ffill()
rf_aligned.index.name = 'date'

print(f"Aligned to {len(rf_aligned)} yfinance trading days")
rf_aligned.head()


Aligned to 2010 yfinance trading days


,risk_free_rate_pct,risk_free_rate_decimal
date,,
2018-01-02,1.44,0.0144
2018-01-03,1.41,0.0141
2018-01-04,1.41,0.0141
2018-01-05,1.39,0.0139
2018-01-08,1.45,0.0145


## Clean CPI (monthly -> forward-filled to daily, for merging with daily context if needed)

In [7]:
cpi_filled = cpi.asfreq('D').ffill()
cpi_filled['cpi_pct_change'] = cpi_filled['cpi_index'].pct_change()
cpi_filled.head()

,cpi_index,cpi_pct_change
date,,
2018-01-01,248.859,NaN
2018-01-02,248.859,0.0
2018-01-03,248.859,0.0
2018-01-04,248.859,0.0
2018-01-05,248.859,0.0


## Save processed data

In [8]:
rf_aligned.to_csv(PROCESSED_DATA_PATH / 'fred_risk_free_rate_processed.csv')
cpi_filled.to_csv(PROCESSED_DATA_PATH / 'fred_cpi_processed.csv')

print("Saved processed FRED data to", PROCESSED_DATA_PATH)

Saved processed FRED data to ..\..\data\processed\fred
